# Classic Component Making (B-3046295)
- Non-modeled components: expense, cat loads, cost of capital, NCOR to AAP, balance
- Retention training: owned by Yinan, not run here

**Before running**
1. Attach the classic environment (Resources must hold `classic_specialty_ltv-10.0.0/` and `ltv_helpers-2.0.0/`)
2. Set `CLV_lakehouse_classic_spl_STG` as the default lakehouse
3. Restart the session after any change to Resources, then run top to bottom

In [ ]:
RUN_TYPE = "prod"
GIT_CHECKOUT = "feature/B-3046295"  # must match the git_checkout label used by the copy pipeline
RUN_VERSION = "2.0.0"               # third arg to make_paths; value taken from Yinan's notebook
PKG_VERSION = "10.0.0"
HELPERS_VERSION = "2.0.0"
EXPECTED_LAKEHOUSE = "CLV_lakehouse_classic_spl_STG"

## Setup

In [ ]:
import sys

import notebookutils

# Resources hold unpacked sdists. The folder uses the PEP 625 distribution name
# (classic_specialty_ltv-<ver>); the import name inside it is classic_spl_ltv.
SPL_ROOT = f"{notebookutils.nbResPath}/env/classic_specialty_ltv-{PKG_VERSION}"
HELPER_ROOT = f"{notebookutils.nbResPath}/env/ltv_helpers-{HELPERS_VERSION}"

for root in (SPL_ROOT, HELPER_ROOT):
    if root not in sys.path:
        sys.path.insert(0, root)

In [ ]:
import importlib
import os

lakehouse = notebookutils.runtime.context.get("defaultLakehouseName")
assert lakehouse == EXPECTED_LAKEHOUSE, f"Default lakehouse is {lakehouse!r}; attach {EXPECTED_LAKEHOUSE}"

print("env Resources:", sorted(os.listdir(f"{notebookutils.nbResPath}/env")))
for root in (SPL_ROOT, HELPER_ROOT):
    assert os.path.isdir(root), f"Missing in environment Resources: {root}"

# A ModuleNotFoundError here names the package missing from the environment YAML
for name in ("loguru", "ltv_helpers.fabric_pipeline_helpers", "classic_spl_ltv.config.run_paths"):
    importlib.import_module(name)

import classic_spl_ltv
import ltv_helpers

for mod, root in ((classic_spl_ltv, SPL_ROOT), (ltv_helpers, HELPER_ROOT)):
    print(f"{mod.__name__:16s} {mod.__file__}")
    assert mod.__file__.startswith(root), f"{mod.__name__} loaded from outside Resources; restart the session"

## Run paths

In [ ]:
from delta.tables import DeltaTable

from classic_spl_ltv.config import run_paths

p = run_paths.make_paths(RUN_TYPE, GIT_CHECKOUT, RUN_VERSION)

BASE = f"Files/data/classic_spl_ltv/{RUN_TYPE}/{GIT_CHECKOUT}"  # must equal "base" in the banner above
print("eltv_pipeline inputs present:", notebookutils.fs.exists(f"{BASE}/eltv_pipeline"))


def check_delta(path):
    """Fail if no Delta table at path; show last write time and shape."""
    assert notebookutils.fs.exists(f"{path}/_delta_log"), f"No Delta table at {path}"
    DeltaTable.forPath(spark, path).history(1).select("timestamp", "operation").show(truncate=False)
    df = spark.read.format("delta").load(path)
    print(f"{path}\nrows={df.count():,}  cols={len(df.columns)}")

## Non-modeled components
### Expense
Reads the Finance Excel file, so it does not depend on the parquet to Delta conversion.

In [ ]:
import classic_spl_ltv.jobs.expense.process_expense_data.process_raw_expense as prexp

prexp.main(spark)

In [ ]:
# Path must match the "Successfully wrote" log line above; timestamp must be today
check_delta(f"{BASE}/expense/processed_expense_df")

### Cat loads, cost of capital, NCOR to AAP, balance
Copy these cells from Yinan's `classic_spl_component_making`, one section per component:
1. The import + `main(spark)` cell, unchanged
2. A check cell: `check_delta(f"{BASE}/<path from the log line>")`

Components that read `eltv_pipeline` inputs may fail until Yinan's parquet to Delta conversion lands. Stop at the first error.

## Retention training
Owned by Yinan. Not run in this notebook.